# Cluster Analysis: Geographic Route Segmentation

This notebook groups the 270 seller pickup points into operationally cohesive geographic zones using K-means clustering. Each cluster represents a logical delivery route that can be covered by a dedicated vehicle fleet.

**Why clustering?** The pickup points span ~55 km × 55 km across the São Paulo metropolitan area. A carrier cannot efficiently serve all points with a single undifferentiated fleet — clustering enables route specialization, shorter per-vehicle distances, and more accurate fleet sizing per zone.

**Feature engineering:** Beyond raw coordinates, the clustering incorporates package volume as a weighted feature. High-volume sellers pull cluster centroids toward them, which ensures that the most operationally demanding points don't end up on the geographic periphery of their assigned cluster.

---

## 1. Setup

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings

sys.path.append('../src')
from logistics_optimizer import LogisticsOptimizer

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

print("LogisticsOptimizer loaded.")

## 2. Load Geocoded Data

We load from the processed Excel file (output of the carrier's operations system, enriched with coordinates from notebook 01). If only the CSV geocoding output is available, an alternative load path is provided.

In [ ]:
try:
    df = pd.read_excel('../data/processed/Dados_logistica.xlsx')
    print(f"Loaded from Excel: {len(df)} rows")
except FileNotFoundError:
    df = pd.read_csv('../data/processed/enderecos_com_coordenadas.csv')
    print(f"Loaded from CSV fallback: {len(df)} rows")

# Normalize column name if needed
volume_col = next((c for c in df.columns if 'VOLUME' in c.upper() and 'PICKUP' in c.upper()), None)
if volume_col and volume_col != 'VOLUME_PICKUP':
    df = df.rename(columns={volume_col: 'VOLUME_PICKUP'})

print(f"Columns: {df.columns.tolist()}")
df.head()

## 3. Exploratory Data Analysis

Before clustering, we characterize the dataset to understand geographic spread, volume distribution, and which areas dominate pickup demand.

In [ ]:
optimizer = LogisticsOptimizer()
optimizer.load_data(df=df)
stats = optimizer.exploratory_analysis()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Volume distribution (log scale highlights the long tail)
axes[0].hist(optimizer.df['VOLUME_PICKUP'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_yscale('log')
axes[0].set_xlabel('Packages per Pickup Point')
axes[0].set_ylabel('Count (log scale)')
axes[0].set_title('Volume Distribution')

# Geographic scatter
sc = axes[1].scatter(
    optimizer.df['LONGITUDE'],
    optimizer.df['LATITUDE'],
    c=optimizer.df['VOLUME_PICKUP'],
    cmap='YlOrRd',
    s=30,
    alpha=0.7
)
plt.colorbar(sc, ax=axes[1], label='Volume')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
axes[1].set_title('Geographic Distribution (color = volume)')

# Top 10 areas by total volume
if 'AREA' in optimizer.df.columns:
    top_areas = optimizer.df.groupby('AREA')['VOLUME_PICKUP'].sum().nlargest(10)
    top_areas.plot(kind='barh', ax=axes[2], color='steelblue')
    axes[2].set_xlabel('Total Volume')
    axes[2].set_title('Top 10 Areas by Volume')
    axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

## 4. Finding Optimal Number of Clusters

Two complementary methods are used to choose `k`:

- **Elbow method:** Plot inertia (within-cluster sum of squares) vs. k. The "elbow" — where the marginal gain from adding another cluster drops sharply — suggests the optimal k.
- **Silhouette score:** Measures how well each point fits its own cluster vs. neighboring clusters. Higher is better. Peak silhouette identifies the k where clusters are most cohesive and well-separated.

When the two methods disagree, we take their average as a practical compromise.

In [ ]:
optimal_k, (K_range, inertias, silhouette_scores) = optimizer.find_optimal_clusters(max_clusters=10)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

K_list = list(K_range)

# Elbow curve
ax1.plot(K_list, inertias, 'bo-', linewidth=2, markersize=6)
ax1.axvline(x=optimal_k, color='red', linestyle='--', label=f'Chosen k={optimal_k}')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia')
ax1.set_title('Elbow Method')
ax1.legend()
ax1.set_xticks(K_list)

# Silhouette scores
ax2.plot(K_list, silhouette_scores, 'gs-', linewidth=2, markersize=6)
ax2.axvline(x=optimal_k, color='red', linestyle='--', label=f'Chosen k={optimal_k}')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Analysis')
ax2.legend()
ax2.set_xticks(K_list)

plt.suptitle(f'Optimal k = {optimal_k} (average of elbow and silhouette suggestions)', fontsize=13)
plt.tight_layout()
plt.show()

## 5. K-means Clustering

**Feature vector:** `[lat_scaled, lon_scaled, volume_normalized × 0.3]`

The 0.3 weight on volume means that geographic proximity dominates the clustering, but high-volume sellers still exert gravitational pull on their cluster centroid. Without this weighting, a cluster could span a large geographic area but contain one dominant seller whose volume skews the fleet estimates.

Configuration: `n_init=20` restarts guard against poor local minima.

In [ ]:
clusters = optimizer.perform_clustering(n_clusters=optimal_k)
clusters

## 6. Cluster Visualization

In [ ]:
# Static scatter plot by cluster
n_clusters = optimizer.df['CLUSTER'].nunique()
palette = cm.get_cmap('tab10', n_clusters)

fig, ax = plt.subplots(figsize=(10, 8))

for cid in sorted(optimizer.df['CLUSTER'].unique()):
    subset = optimizer.df[optimizer.df['CLUSTER'] == cid]
    ax.scatter(
        subset['LONGITUDE'],
        subset['LATITUDE'],
        s=subset['VOLUME_PICKUP'] / 5 + 10,
        alpha=0.6,
        label=f"Cluster {cid} ({len(subset)} pts)",
        color=palette(cid)
    )

# Centroids
for _, c in clusters.iterrows():
    ax.scatter(
        c['centroid_lon'], c['centroid_lat'],
        marker='*', s=300, color='black', zorder=5
    )

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'Pickup Point Clusters (k={n_clusters}) — São Paulo Metro Area')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Interactive Folium map (renders inline in Jupyter)
mapa = optimizer.create_interactive_map()
mapa